# Mount Drive & chọn GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi   # kiểm tra A100

Mounted at /content/drive
Wed Aug 19 05:33:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

# Giải nén dataset

In [ ]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Datasets/CAERS_train.zip"

!mkdir -p /content/CAER-S
!unzip -q "$DRIVE_ZIP_PATH" -d /content/CAER-S
!ls /content/CAER-S/train   # phải thấy 7 folder: Angry Disgust Fear Happy Neutral Sad Surprise

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise


# Clone repo

In [ ]:
!git clone https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 335, done.
remote: Counting objects: 100% (335/335), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 335 (delta 257), reused 324 (delta 249), pack-reused 0 (from 0)
Receiving objects: 100% (335/335), 30.25 MiB | 34.04 MiB/s, done.
Resolving deltas: 100% (257/257), done.
/content/Exp-CLIP


## Pull code


In [ ]:
# %cd /content/Exp-CLIP
# !git pull

# !git checkout -- train.py
!git pull

Already up to date.


# Cài dependencies

 Tạo venv Python 3.10

In [ ]:
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.9 MB/s eta 0:00:00


Cài torch (khớp CUDA) + dependencies + lavis vào venv đó

In [ ]:
!/content/venv310/bin/pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q salesforce-lavis
!/content/venv310/bin/pip install -q "numpy<2"   # phải cài SAU lavis, vì lavis kéo theo numpy 2.x
!/content/venv310/bin/pip install -q matplotlib scikit-learn

# Sửa đường dẫn dataset trong train.py

In [ ]:
import re
path = "train.py"
content = open(path).read()
content = re.sub(
    r"train_data_file_path = '.*'",
    "train_data_file_path = '/content/CAER-S/train'",
    content
)
open(path, "w").write(content)
!grep -n "train_data_file_path =" train.py

70:    train_data_file_path = '/content/CAER-S/train'


In [ ]:
!du -sh /content/* 2>/dev/null | sort -rh | head -15
!du -sh /root/.cache/* ~/.cache/* 2>/dev/null | sort -rh | head -15
!du -sh /usr/local/lib/python3.10/dist-packages/* 2>/dev/null | sort -rh | head -15
!du -sh /content/venv310/lib/python3.10/site-packages/* 2>/dev/null | sort -rh | head -15

9.9G	/content/drive
9.2G	/content/CAER-S
5.7G	/content/venv310
68M	/content/Exp-CLIP
43G	/root/.cache/huggingface
890M	/root/.cache/clip
56M	/root/.cache/node-gyp
36K	/root/.cache/matplotlib
8.0K	/root/.cache/antigravity
4.0K	/root/.cache/torch
3.9G	/content/venv310/lib/python3.10/site-packages/torch
185M	/content/venv310/lib/python3.10/site-packages/triton
151M	/content/venv310/lib/python3.10/site-packages/pyarrow
112M	/content/venv310/lib/python3.10/site-packages/spacy
109M	/content/venv310/lib/python3.10/site-packages/scipy
101M	/content/venv310/lib/python3.10/site-packages/pycocoevalcap
69M	/content/venv310/lib/python3.10/site-packages/cv2
66M	/content/venv310/lib/python3.10/site-packages/plotly
65M	/content/venv310/lib/python3.10/site-packages/pandas
64M	/content/venv310/lib/python3.10/site-packages/cmake
61M	/content/venv310/lib/python3.10/site-packages/opencv_python_headless.libs
57M	/content/venv310/lib/python3.10/site-packages/sympy
53M	/content/venv310/lib/python3.10/site-pac

In [ ]:
!nproc

12


#  Chạy train (đúng cấu hình runner.sh, chỉ backbone L14)

## seed 1

In [ ]:
import os
os.environ["JOB_ID"] = "expclip_l14_run1"

!/content/venv310/bin/python train.py \
  --workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 \
  --lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed 1 \
  --job-id $JOB_ID \
  --instruction "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image." \
  --load-model "CLIP_L14"

Training date:  26-08-19 06:51
************************
workers = 8
epochs = 5
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 10
milestones = [30]
seed = 1
job_id = expclip_l14_run1
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
************************
/content/venv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(

********************0********************
Epoch: [0][ 0/95]	Loss 6.5922 (6.5922)	Accuracy  0.195 ( 0.195)
Epoch: [0][10/95]	Loss 6.3751 (6.4943)	Accuracy  0.195 ( 0.142)
Epoch: [0][20/95]	Loss 6.2430 (6.4107)	Accuracy  0.586 ( 0.251)
Epoch: [0][30/95]	L

## seed 2

In [ ]:
import os
os.environ["JOB_ID"] = "expclip_l14_seed2"

!/content/venv310/bin/python train.py \
  --workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 \
  --lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed 2 \
  --job-id $JOB_ID \
  --instruction "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image." \
  --load-model "CLIP_L14" \
  --extra-checkpoint-dir "/content/drive/MyDrive/ExpCLIP_output"

Training date:  26-08-19 08:40
************************
workers = 8
epochs = 5
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 10
milestones = [30]
seed = 2
job_id = expclip_l14_seed2
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output
************************
/content/venv310/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Traceback (most recent call last):
  File "/content/Exp-CLIP/train.py", line 316, in <module>
    main()
  File "/content/Exp-CLIP/train.py", line 74, in main
    model = ExpCLIP_Train(args)
 

## seed 3

In [ ]:
import os
os.environ["JOB_ID"] = "expclip_l14_seed3"

!/content/venv310/bin/python train.py \
  --workers 8 --epochs 5 --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 \
  --lr 1e-3 --weight-decay 1e-4 --momentum 0.9 --print-freq 10 --milestones 30 --seed 3 \
  --job-id $JOB_ID \
  --instruction "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image." \
  --load-model "CLIP_L14" \
  --extra-checkpoint-dir "/content/drive/MyDrive/ExpCLIP_output"

# Backup checkpoint + log về Drive ngay khi xong


In [ ]:
!mkdir -p /content/drive/MyDrive/ExpCLIP_output
!cp checkpoint/expclip_l14_run1-model.pth /content/drive/MyDrive/ExpCLIP_output/
!cp log/expclip_l14_run1-log.* /content/drive/MyDrive/ExpCLIP_output/

# Xác nhận checkpoint đã lưu

In [ ]:
!ls -lh checkpoint/expclip_l14_run1-model.pth

-rw-r--r-- 1 root root 13M Aug 19 07:38 checkpoint/expclip_l14_run1-model.pth


# Giải nén test set

In [ ]:
!mkdir -p /content/static_test_sets
!unzip -q /content/drive/MyDrive/Datasets/static_test_sets.zip -d /content/static_test_sets
!ls /content/static_test_sets
!ls /content/static_test_sets/RAFDB/test   # phải thấy 0_Happiness ... 6_Fear

AffectNet7  AffectNet8	FERPlus  RAFDB
0_Happiness  1_Sadness	2_Neutral  3_Anger  4_Surprise	5_Disgust  6_Fear


# Chạy test

## seed 1

In [ ]:
!/content/venv310/bin/python test.py \
  --load-model "CLIP_L14" \
  --job-id "expclip_l14_run1" \
  --data-root /content/static_test_sets

************************
load_model = CLIP_L14
job_id = expclip_l14_run1
data_root = /content/static_test_sets
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 41.78/41.07
************************* AffectNet7
UAR/WAR: 37.94/37.94
************************* AffectNet8
UAR/WAR: 32.40/32.40
************************* FERPlus
UAR/WAR: 44.82/45.17
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 55.96/55.08
************************* AffectNet7
UAR/WAR: 43.37/43.37
************************* AffectNet8
UAR/WAR: 37.45/37.45
************************* FERPlus
UAR/WAR: 38.48/47.89
***********************************

## seed 2

In [ ]:
!/content/venv310/bin/python test.py --load-model "CLIP_L14" --job-id "expclip_l14_seed2" --data-root /content/static_test_sets

## seed 3

In [ ]:
!/content/venv310/bin/python test.py --load-model "CLIP_L14" --job-id "expclip_l14_seed3" --data-root /content/static_test_sets